# emogist: final test (Ministral 3 3B)

This notebook runs the trained adapter once on the 100 test sentences. Ministral 3 3B is the **comparison model**: its score is reported next to Qwen, and the choice of Qwen does not change.

Nothing is trained here. **Before you start:** open **Runtime > Change runtime type**, select **T4 GPU**, and save. Then run the cells from top to bottom, one at a time.


## 1. Install the libraries

The same versions as the training notebook. If Colab asks you to restart the session afterwards, accept, then continue from step 2.


In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

## 2. Settings


In [ ]:
import os
import torch

RUN_NAME = "ministral3b_full"
BASE_MODEL = "unsloth/Ministral-3-3B-Instruct-2512"
MAX_LENGTH = 256
SYSTEM_PROMPT = "Convert the sentence into a short emoji sentence that keeps its gist. Answer with emojis only."
DRIVE_DIR = f"/content/drive/MyDrive/emogist/run_{RUN_NAME}"
ZIP_NAME = f"run_{RUN_NAME}.zip"
RUN_DIR = f"/content/run_{RUN_NAME}"
TEST_FILE = "test_sentences.csv"

assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))


## 3. Get the trained adapter and the test sentences

Colab asks for access to your Google Drive, where the training notebook saved `run_ministral3b_full.zip`. If the zip isn't found, it asks you to upload it from `04_fine_tuning/runs/`.

Then choose `data/test_sentences.csv` from your repo.


In [ ]:
import json
import shutil
from google.colab import drive, files

drive.mount("/content/drive")
zip_path = f"{DRIVE_DIR}/{ZIP_NAME}"
if not os.path.exists(zip_path):
    print("Not found in Drive. Please select", ZIP_NAME)
    files.upload()
    zip_path = ZIP_NAME
assert os.path.exists(zip_path), f"Missing {ZIP_NAME}"
shutil.unpack_archive(zip_path, RUN_DIR)

with open(f"{RUN_DIR}/settings.json") as f:
    settings = json.load(f)
assert settings["base_model"] == BASE_MODEL and settings["system_prompt"] == SYSTEM_PROMPT, "Wrong run"
print(f"Adapter from epoch {settings['best_epoch']:.0f}")

if not os.path.exists(TEST_FILE):
    print("Please select", TEST_FILE)
    files.upload()
assert os.path.exists(TEST_FILE), f"Missing {TEST_FILE}"
print("Ready")


## 4. Load the model and its adapter


In [ ]:
from transformers import AutoTokenizer
from unsloth import FastVisionModel

model, _ = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)
text_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Model loaded")

from peft import PeftModel

model = PeftModel.from_pretrained(model, f"{RUN_DIR}/adapter")
FastVisionModel.for_inference(model)
print("Adapter loaded")


## 5. Check the adapter on the dev set

This regenerates the 100 dev outputs and compares them with the ones saved after training. If they don't match, the notebook stops **before** touching the test set. In that case, send me the output.


In [ ]:
def to_messages(sentence, emojis=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": sentence}]
    if emojis is not None:
        messages.append({"role": "assistant", "content": emojis})
    return messages

def to_emojis(sentence):
    prompt = text_tokenizer.apply_chat_template(to_messages(sentence), tokenize=False,
                                                add_generation_prompt=True)
    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False, use_cache=True,
                                eos_token_id=text_tokenizer.eos_token_id,
                                pad_token_id=text_tokenizer.pad_token_id)
    answer = output[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(answer, skip_special_tokens=True).strip()


import csv

with open(f"{RUN_DIR}/dev_outputs_{RUN_NAME}.csv", newline="", encoding="utf-8") as f:
    saved = list(csv.DictReader(f))
same = sum(to_emojis(row["sentence"]) == row["output"] for row in saved)
print(f"{same}/{len(saved)} dev outputs identical to the training run")
assert len(saved) == 100 and same >= 95, "The adapter doesn't reproduce its dev outputs: stop and send me this output"
dev_check_passed = True


## 6. Run the test

Run this cell **once**. Don't change anything in the model after seeing the result.


In [ ]:
assert globals().get("dev_check_passed"), "Run step 5 first"

with open(TEST_FILE, newline="", encoding="utf-8") as f:
    test = list(csv.DictReader(f))
assert len(test) == 100, f"Expected 100 test sentences, found {len(test)}"

for i, row in enumerate(test, 1):
    row["output"] = to_emojis(row["sentence"])
    if i % 25 == 0:
        print(f"{i}/{len(test)}")

outputs_file = f"test_outputs_{RUN_NAME}.csv"
with open(outputs_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "lang", "sentence", "output"], extrasaction="ignore")
    writer.writeheader()
    writer.writerows(test)
shutil.copy(outputs_file, DRIVE_DIR)
print("Saved", outputs_file)


## 7. Download the result

Put `test_outputs_ministral3b_full.csv` in `04_fine_tuning/runs/run_ministral3b_full/`, tell me when it's there, then disconnect the runtime.


In [ ]:
files.download(outputs_file)
